# 05 - Weather + email (a "reactive" action)

Combines `get_weather` (from notebook 04) with a new `send_email` tool that has a **real side effect** - it actually sends mail via Gmail SMTP. This is the step from "agent that answers" to "agent that does something in the world."

**Setup - Gmail App Password (not your normal password):**
1. Enable 2-Step Verification: https://myaccount.google.com/security
2. Create an App Password: https://myaccount.google.com/apppasswords (choose "Mail", name it anything, copy the 16-character password)
3. Enter your Gmail address and that App Password when prompted below - never your real Google account password.

## 1. Install dependencies

In [ ]:
%pip install -q anthropic requests

## 2. Set your credentials

The App Password uses `getpass` (hidden prompt, never saved into the notebook).

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")
if not os.environ.get("EMAIL_ADDRESS"):
    os.environ["EMAIL_ADDRESS"] = input("Enter your Gmail address: ").strip()
if not os.environ.get("EMAIL_APP_PASSWORD"):
    os.environ["EMAIL_APP_PASSWORD"] = getpass("Enter your Gmail App Password: ")

# Only needed if you hit: "anthropic-workspace-id is required when
# authenticating with an identity-linked API key" - leave blank to skip.
if not os.environ.get("ANTHROPIC_WORKSPACE_ID"):
    _workspace_id = input("ANTHROPIC_WORKSPACE_ID (leave blank if not needed): ").strip()
    if _workspace_id:
        os.environ["ANTHROPIC_WORKSPACE_ID"] = _workspace_id

## 3. Tools, the agent loop, and the `Agent` class

Same cost cap and turn-collapsing core as the other notebooks. Note the system prompt explicitly tells the model to only send email when asked - a tool with a real side effect shouldn't fire on its own initiative.

In [ ]:
import datetime
import json
import os

import anthropic

MODEL = "claude-haiku-4-5"
MAX_TOKENS = int(os.environ.get("AGENT_MAX_TOKENS", "1024"))

# claude-haiku-4-5 pricing, $/1M tokens - update if you switch models.
INPUT_COST_PER_MTOK = 1.00
OUTPUT_COST_PER_MTOK = 5.00

# Hard spending cap for this notebook kernel session.
MAX_COST_USD = float(os.environ.get("AGENT_MAX_COST_USD", "0.20"))


class BudgetExceededError(RuntimeError):
    pass
import smtplib
from email.mime.text import MIMEText

import requests

SYSTEM_PROMPT = (
    "You are a helpful assistant with get_weather and send_email tools. "
    "Use get_weather for current conditions. Only call send_email when the "
    "user explicitly asks you to send or email something - never send "
    "email on your own initiative. Otherwise reply directly."
)

TOOLS = [
    {
        "name": "get_weather",
        "description": "Get current weather conditions for a location by city name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, optionally with country, e.g. 'Paris, France'.",
                },
            },
            "required": ["location"],
        },
    },
    {
        "name": "send_email",
        "description": "Send a plain-text email via Gmail to a recipient.",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email address."},
                "subject": {"type": "string", "description": "Email subject line."},
                "body": {"type": "string", "description": "Plain-text email body."},
            },
            "required": ["to", "subject", "body"],
        },
    },
]

# WMO weather interpretation codes (used by Open-Meteo's weather_code field).
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    56: "Light freezing drizzle", 57: "Dense freezing drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    66: "Light freezing rain", 67: "Heavy freezing rain",
    71: "Slight snow fall", 73: "Moderate snow fall", 75: "Heavy snow fall",
    77: "Snow grains",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


def _get_with_retry(url: str, params: dict, attempts: int = 2):
    last_exc = None
    for attempt in range(attempts):
        try:
            resp = requests.get(url, params=params, timeout=20)
            resp.raise_for_status()
            return resp
        except requests.RequestException as exc:
            last_exc = exc
    raise last_exc


def get_weather(location: str) -> str:
    """Free, no-API-key weather lookup via Open-Meteo."""
    try:
        geo_resp = _get_with_retry(
            "https://geocoding-api.open-meteo.com/v1/search",
            {"name": location, "count": 1},
        )
        geo_data = geo_resp.json()
    except requests.RequestException as exc:
        return f"Error: location lookup failed ({exc})"

    results = geo_data.get("results")
    if not results:
        return f"Error: could not find a location matching '{location}'."
    place = results[0]

    try:
        weather_resp = _get_with_retry(
            "https://api.open-meteo.com/v1/forecast",
            {
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,weather_code",
            },
        )
        weather_data = weather_resp.json()
    except requests.RequestException as exc:
        return f"Error: weather lookup failed ({exc})"

    current = weather_data.get("current", {})
    units = weather_data.get("current_units", {})
    condition = WMO_CODES.get(current.get("weather_code"), "Unknown conditions")
    place_label = place["name"] + ((", " + place["country"]) if place.get("country") else "")

    return (
        "Weather in " + place_label + ": " + condition + ", "
        + str(current.get("temperature_2m")) + units.get("temperature_2m", "\u00b0C")
        + ", wind " + str(current.get("wind_speed_10m")) + " " + units.get("wind_speed_10m", "km/h")
    )


def send_email(to: str, subject: str, body: str) -> str:
    """Send via Gmail SMTP using an App Password (not your normal Gmail
    password - see https://myaccount.google.com/apppasswords, requires
    2-Step Verification enabled)."""
    sender = os.environ.get("EMAIL_ADDRESS")
    app_password = os.environ.get("EMAIL_APP_PASSWORD")
    if not sender or not app_password:
        return "Error: EMAIL_ADDRESS / EMAIL_APP_PASSWORD not set."

    msg = MIMEText(body)
    msg["Subject"] = subject
    msg["From"] = sender
    msg["To"] = to

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as server:
            server.login(sender, app_password)
            server.sendmail(sender, [to], msg.as_string())
        return f"Email sent to {to}."
    except smtplib.SMTPException as exc:
        return f"Error: failed to send email ({exc})"


def execute_tool(name: str, tool_input: dict) -> str:
    if name == "get_weather":
        return get_weather(tool_input["location"])
    if name == "send_email":
        return send_email(tool_input["to"], tool_input["subject"], tool_input["body"])
    return f"Error: unknown tool '{name}'"


MAX_PAUSE_RESUMES = 10


def build_client() -> anthropic.Anthropic:
    """Some API keys (personal keys not scoped to one workspace) require an
    anthropic-workspace-id header on every request - see
    https://platform.claude.com/docs/en/manage-claude/authentication#select-a-workspace.
    Set ANTHROPIC_WORKSPACE_ID if you hit: 'anthropic-workspace-id is
    required when authenticating with an identity-linked API key'."""
    workspace_id = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    if workspace_id:
        return anthropic.Anthropic(
            default_headers={"anthropic-workspace-id": workspace_id}
        )
    return anthropic.Anthropic()


class Agent:
    """A minimal conversational agent that can call tools in a loop."""

    def __init__(self, client: anthropic.Anthropic | None = None):
        self.client = client or build_client()
        self.messages: list[dict] = []
        self.total_cost_usd = 0.0

    def send(self, user_input: str) -> str:
        turn_start = len(self.messages)
        self.messages.append({"role": "user", "content": user_input})

        resumes = 0
        while True:
            if self.total_cost_usd >= MAX_COST_USD:
                raise BudgetExceededError(
                    f"Session cost ${self.total_cost_usd:.4f} has reached the "
                    f"${MAX_COST_USD:.4f} cap (AGENT_MAX_COST_USD). Raise the "
                    "cap or start a new session to continue."
                )

            response = self.client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                tools=TOOLS,
                messages=self.messages,
            )
            self.total_cost_usd += (
                response.usage.input_tokens * INPUT_COST_PER_MTOK
                + response.usage.output_tokens * OUTPUT_COST_PER_MTOK
            ) / 1_000_000
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "pause_turn":
                resumes += 1
                if resumes > MAX_PAUSE_RESUMES:
                    break
                continue

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )
            self.messages.append({"role": "user", "content": tool_results})

        reply = "".join(
            block.text for block in response.content if block.type == "text"
        )
        self.messages[turn_start:] = [
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": reply},
        ]
        return reply


## 4. Create the agent

In [ ]:
agent = Agent()
print(f"Agent ready (cap ${MAX_COST_USD:.4f} for this kernel session)")

## 5. Try it

Replace `you@example.com` with your own address before running - this really sends an email.

In [ ]:
reply = agent.send(
    "What's the weather in Tokyo, and send it to you@example.com "
    "with the subject 'Tokyo Weather'."
)
print(reply)
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

## 6. Optional: interactive chat loop

Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    try:
        reply = agent.send(user_input)
    except BudgetExceededError as exc:
        print(f"Agent: [stopped] {exc}")
        break
    print(f"Agent: {reply}  (session cost so far: ~${agent.total_cost_usd:.4f})")
